# PikSign GPU Server (Colab)

This notebook runs the PikSign GPU server on Google Colab.
It exposes LEAT, ContentAnalyzer, and SemanticDrift via FastAPI + ngrok.

**Steps:**
1. Clone repo and install dependencies
2. Download model weights
3. Configure ngrok
4. Start server and get your URL
5. Paste the URL into PikSign app sidebar

In [ ]:
# Cell 1: Clone repo and install dependencies
!git clone https://github.com/Arun973150/piksign.git
%cd piksign

# Install GPU torch (already on Colab, but ensure latest)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# Install FastAPI + ngrok + other deps
!pip install -q fastapi uvicorn pyngrok Pillow numpy scipy opencv-python-headless gdown

print("\n[OK] Dependencies installed")

In [ ]:
# Cell 2: Download encoder weights from Google Drive
# Downloads all LEAT encoder weights (~4.6 GB total)

import os
import gdown

weights_dir = 'piksign/protection/weights'
os.makedirs(weights_dir, exist_ok=True)

# Download entire weights folder from Google Drive
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1JDtZrTvbdBZT_wZxX9XIEpGhX99sIX-c"

print("Downloading weights from Google Drive...")
gdown.download_folder(DRIVE_FOLDER_URL, output=weights_dir, quiet=False)

# Verify weights
print("\n" + "=" * 50)
print("Weight status:")
print("=" * 50)
files = os.listdir(weights_dir)
total_size = 0
for f in sorted(files):
    size = os.path.getsize(os.path.join(weights_dir, f)) / (1024*1024)
    total_size += size
    print(f"  [OK] {f}: {size:.1f} MB")
print(f"\nTotal: {len(files)} files, {total_size/1024:.1f} GB")

In [ ]:
# Cell 3: Configure ngrok
# Get your free authtoken at: https://dashboard.ngrok.com/get-started/your-authtoken

from pyngrok import ngrok
import getpass

NGROK_TOKEN = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)
print("[OK] ngrok configured")

In [ ]:
# Cell 4: Start GPU server + ngrok tunnel
import subprocess
import time
import requests
from pyngrok import ngrok

# Start FastAPI server — do NOT pipe stdout so logs are visible and buffer can't block
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'colab_server:app', '--host', '0.0.0.0', '--port', '8000'],
)

# Wait for server to be ready (poll /health instead of fixed sleep)
print("Loading models into GPU... (may take 2-3 minutes)")
deadline = time.time() + 300  # 5-minute timeout
ready = False
while time.time() < deadline:
    time.sleep(5)
    try:
        r = requests.get('http://localhost:8000/api/v1/health', timeout=3)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    print("  ...still loading")

if not ready:
    raise RuntimeError("Server did not start within 5 minutes. Check the logs above.")

# Create ngrok tunnel
public_url = ngrok.connect(8000, "http")

print("\n" + "=" * 60)
print("  PIKSIGN GPU SERVER READY")
print("=" * 60)
print(f"\n  Paste this URL into PikSign sidebar:")
print(f"  {public_url.public_url}")
print(f"\n" + "=" * 60)

In [ ]:
# Cell 5: Keep alive + monitor
# Run this cell to keep the server alive.
# The Colab session will stay active as long as this cell runs.

import time
import requests

print("Server running. Press stop button to shut down.")
print(f"URL: {public_url.public_url}")
print()

try:
    while True:
        time.sleep(60)
        try:
            r = requests.get('http://localhost:8000/api/v1/health', timeout=5)
            if r.status_code == 200:
                info = r.json()
                print(f"[{time.strftime('%H:%M')}] OK - GPU: {info.get('gpu', '?')}, "
                      f"Encoders: {len(info.get('encoders_loaded', []))}")
            else:
                print(f"[{time.strftime('%H:%M')}] Warning: status {r.status_code}")
        except Exception as e:
            print(f"[{time.strftime('%H:%M')}] Health check failed: {e}")
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.disconnect(public_url.public_url)
    server_proc.terminate()
    print("[OK] Server stopped.")